## Camada 1 - aprendizado zero-shot

Vamos começar pelo mais simples: uma chamada ao modelo, zero
exemplos no prompt. O nome disso na literatura: *zero-shot*.

A tarefa: classificar reclamações de um e-commerce em cinco
categorias fechadas. A pergunta desta camada: onde o modelo
escorrega recebendo só a instrução?


In [1]:
# Setup: dados, credenciais e o helper agnóstico (llm.py)
import os, pathlib, sys

sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
from llm import chat

print(f"Modelo: {os.environ.get('LLM_MODEL', '?')}")

df = pd.read_csv('../data/reclamacoes.csv')
df.head(3)


Modelo: deepseek-v4.1-flash


,texto,categoria
0,O rastreador do pedido não atualiza há duas se...,entrega
1,Cobraram o frete duas vezes na minha fatura do...,cobranca
2,O produto chegou com a embalagem amassada e se...,produto


In [2]:
# Montagem do prompt: instrução + lista fechada de categorias + texto.
# O prompt zero-shot também está em 02 e bonus_evals; se alterar,
# replique nos três para manter o placar comparável.
# Pedir 'responda só com a categoria' não é estética: saída previsível
# é o que permite comparar execuções lado a lado.
CATEGORIAS = 'entrega | cobranca | produto | atendimento | app'

def prompt_zero_shot(texto: str) -> str:
    return (
        'Classifique a reclamacao em UMA das categorias:\n'
        f'{CATEGORIAS}\n\n'
        f'Reclamacao: "{texto}"\n\n'
        'Responda apenas com a categoria, em minusculas, e nada mais.'
    )


In [ ]:
# temperature=0 reduz o acaso: mesma entrada tende à mesma saída.
# (temperature é o único parâmetro usado aqui; os outros entram na camada 2)
INDICES = [15, 4, 16, 10, 17, 1]  # índices 15 e 16 são armadilhas multimotivo: erram juntas
# INDICES = [3, 10, 1] 

for i in INDICES:
    linha = df.iloc[i]
    pred = chat(
                [
                    {
                     'role': 'user', 
                     'content': prompt_zero_shot(linha['texto'])
                     }
                ], temperature=0).strip().lower()
    
    ok = 'OK' if pred == linha['categoria'] else 'ERROU'
    
    print(f'[{ok}] {linha["texto"][:55]}...')
    print(f'      esperado: {linha["categoria"]} | veio: {pred}\n')


[ERROU] O atendente confirmou o reembolso do produto que nunca ...
      esperado: entrega | veio: cobranca

[ERROU] O desconto do cupom do app sumiu no checkout e a correç...
      esperado: cobranca | veio: app

[OK] O aplicativo trava na tela de pagamento e não deixa fec...
      esperado: app | veio: app

[OK] Meu pedido voltou pro remetente sem aviso e sem eu ter ...
      esperado: entrega | veio: entrega



## Fechamento da camada 1

O zero-shot não erra porque o modelo é ruim: erra porque
recebeu pouco contexto. Os índices 15 e 16 são multimotivo -
ganchos de mais de uma categoria - e ele agarrou a palavra
mais óbvia, perdendo a causa real.

O conserto vem na camada 2, sem trocar de modelo.
